# boolean-mask-identity-replace — ex7: padded-sequence mean ignoring pad positions

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `boolean-mask-identity-replace`. Running the final beacon cell reports progress against the `Numpy: Indexing and selection` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Indexing and selection` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`boolean-mask-identity-replace`** (exercise 7). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "boolean-mask-identity-replace"
DD_SUBTOPIC = "Numpy: Indexing and selection"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Mask & substitute — quick refresher

**Build a mask.** Any comparison returns a `dtype=bool` tensor of the same shape: `x < 0`, `x.abs() < eps`, `(x > 0) & (x < 1)`. Combine with `&`, `|`, `~`.

**Write through a mask.** `y[mask] = value` modifies in place. Scalars broadcast; tensor values must match the shape of `y[mask]` after broadcasting. Always `clone()` first if the function must not mutate its input.

**The dangerous case.** When a mask is the *wrong* shape, indexing can silently collapse axes or pick the wrong cells. Always check `mask.sum()` and `mask.shape` before trusting the result.

### Exercise 7 — padded-sequence mean ignoring pad positions

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Compute per-sequence mean pooling over padded (B, T, D) inputs using a padding mask.
> Keywords: padding-mask, masked-mean, broadcast-mask, axis-reduce, transformer-pooling
> ```

**KCs targeted:** `broadcast-mask-over-feature-axis`, `masked-sum-then-normalize`, `guard-against-zero-length`

Given a padded batch `X` of shape `(B, T, D)` and a padding mask `pad_mask` of shape `(B, T)` where `True` marks **real (non-pad)** positions, implement `ex7_masked_mean(X, pad_mask)` to return a tensor of shape `(B, D)` where each row is the *mean of the real positions only* — pad positions must not contribute, and the divisor must be the per-sequence real-length, not `T`.

**Multi-step debug.** Before returning, `print(...)` four things in this order so you can introspect the pipeline:
1. `pad_mask.shape` and `pad_mask.sum(dim=1)` — per-sequence real-length
2. The expanded-mask shape after `pad_mask.unsqueeze(-1)` — should be `(B, T, 1)`
3. The masked-sum shape — `(B, D)`
4. The divisor — `pad_mask.sum(dim=1, keepdim=True).clamp(min=1).float()` — shape `(B, 1)`, with the `clamp(min=1)` guard so an all-pad row doesn't divide by zero.

Verify by hand: for a single sequence of 3 real tokens and 2 pad tokens, the output should equal the unmasked mean of the first 3 rows.

In [ ]:
def ex7_masked_mean(X: Tensor, pad_mask: Tensor) -> Tensor:
    """Per-sequence mean of real positions in a padded (B, T, D) batch.

    X        : (B, T, D)
    pad_mask : (B, T)  True = real token, False = pad
    returns  : (B, D)  per-sequence mean over real positions only
    """
    raise NotImplementedError()


def _test_ex7():
    # Hand-checked case — 1 sequence, 5 timesteps, 3 real + 2 pad, D = 2
    X = t.tensor([[
        [1.0, 10.0],
        [2.0, 20.0],
        [3.0, 30.0],
        [99.0, 99.0],  # pad — must NOT contribute
        [99.0, 99.0],  # pad
    ]])
    pad_mask = t.tensor([[True, True, True, False, False]])
    out = ex7_masked_mean(X, pad_mask)
    assert out.shape == (1, 2), f'expected (1, 2), got {tuple(out.shape)}'
    expected = t.tensor([[2.0, 20.0]])  # (1+2+3)/3, (10+20+30)/3
    assert t.allclose(out, expected, atol=1e-6), f'expected {expected.tolist()}, got {out.tolist()}'

    # Variable-length batch
    B, T, D = 3, 4, 2
    X2 = t.tensor([
        [[1., 1.], [1., 1.], [0., 0.], [0., 0.]],  # 2 real, mean = [1, 1]
        [[2., 4.], [4., 8.], [6., 12.], [0., 0.]],  # 3 real, mean = [4, 8]
        [[5., 0.], [0., 0.], [0., 0.], [0., 0.]],  # 1 real, mean = [5, 0]
    ])
    pad_mask2 = t.tensor([
        [True, True, False, False],
        [True, True, True, False],
        [True, False, False, False],
    ])
    out2 = ex7_masked_mean(X2, pad_mask2)
    exp2 = t.tensor([[1., 1.], [4., 8.], [5., 0.]])
    assert t.allclose(out2, exp2, atol=1e-6), f'expected {exp2}, got {out2}'

    # Edge case — all-pad sequence must not NaN. Mean should be 0 (divisor clamped to 1)
    X3 = t.tensor([[[5., 5.], [5., 5.]]])
    pad3 = t.tensor([[False, False]])
    out3 = ex7_masked_mean(X3, pad3)
    assert out3.shape == (1, 2)
    assert t.isfinite(out3).all(), f'all-pad sequence produced non-finite output: {out3}'
    assert t.allclose(out3, t.zeros(1, 2), atol=1e-6), f'all-pad must yield zeros, got {out3}'
    _dd_passed.add('ex7')
    print("ex7 ✓")

_test_ex7()

<details><summary>Solution</summary>

```python
def ex7_masked_mean(X: Tensor, pad_mask: Tensor) -> Tensor:
    # 1. Per-sequence real-length
    lengths = pad_mask.sum(dim=1)              # (B,)
    print(f'pad_mask.shape = {tuple(pad_mask.shape)}, lengths = {lengths.tolist()}')

    # 2. Expand mask to (B, T, 1) so it broadcasts over D
    m = pad_mask.unsqueeze(-1).to(X.dtype)     # (B, T, 1)
    print(f'expanded mask shape = {tuple(m.shape)}')

    # 3. Masked sum — pad positions contribute 0
    s = (X * m).sum(dim=1)                     # (B, D)
    print(f'masked sum shape    = {tuple(s.shape)}')

    # 4. Divisor with all-pad guard
    div = lengths.clamp(min=1).unsqueeze(-1).to(X.dtype)   # (B, 1)
    print(f'divisor shape       = {tuple(div.shape)}, divisor = {div.squeeze(-1).tolist()}')
    return s / div
```

**Why `unsqueeze(-1)` and not `unsqueeze(1)`.** Right-align: `(B, T, D)` vs `(B, T, 1)` works (broadcast the trailing 1 over D). `(B, T)` alone would right-align as `(_, B, T)` and fail. The mental model is 'add a feature axis to the mask so it lines up with the feature axis of the data'.

**Why clamp the divisor.** An all-pad row has `lengths[i] = 0`. Dividing by 0 gives NaN, which then poisons every downstream operation. `clamp(min=1)` returns 0/1 = 0 for that row, which is a sensible default — the upstream loss should mask out the all-pad row anyway, but defending in depth is cheap.

**Real-world use.** This is exactly how BERT-style sentence-mean pooling, HuggingFace `mean_pooling`, and many sequence-level classifiers compute their pooled representation.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex7',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()